In [ ]:
import torch
from diffusion.approaches.matching.prob_paths import (
    GaussianCondProbPath,
    LinearAlpha,
    LinearBeta,
)
from diffusion.approaches.matching.flow_trainer import FlowTrainer
from diffusion.sampleables.mnist_sampleable import MNISTSampleable
from diffusion.architectures.backbones.res_unet import ResUnet

In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(device)

mps


In [ ]:
sampeable = MNISTSampleable(train=True)
val_sampeable = MNISTSampleable(train=False)

path = GaussianCondProbPath(
    p_data=sampeable,
    p_simple_shape=sampeable.shape,
    alpha=LinearAlpha(),
    beta=LinearBeta(),
).to(device)

val_path = GaussianCondProbPath(
    p_data=val_sampeable,
    p_simple_shape=sampeable.shape,
    alpha=LinearAlpha(),
    beta=LinearBeta(),
).to(device)

backbone = ResUnet(
    in_channels=1,
    channels=[16, 32, 64],
    num_classes=sampeable.num_classes,
    t_dim=64,
    y_dim=32,
    cond_dim=64,
).to(device)

trainer = FlowTrainer(
    path=path,
    val_path=val_path,
    backbone=backbone,
    null_class=sampeable.num_classes,
)

In [ ]:
trainer.train(
    num_epochs=15,
    device=device,
    lr=3e-4,
    batch_size=64,
    steps_per_epoch=500,
    validate=True,
)

2025-10-11 13:03:44,039 - flow-matching - INFO - Training model with size: 2.734 MiB
Epoch 0/5: 100%|██████████| 500/500 [00:38<00:00, 13.00it/s, loss=0.248149]
2025-10-11 13:04:23,475 - flow-matching - INFO - ['val_loss: 0.183798']
Epoch 1/5: 100%|██████████| 500/500 [00:37<00:00, 13.27it/s, loss=0.163153]
2025-10-11 13:05:02,075 - flow-matching - INFO - ['val_loss: 0.153746']
Epoch 2/5: 100%|██████████| 500/500 [00:37<00:00, 13.21it/s, loss=0.152046]
2025-10-11 13:05:40,835 - flow-matching - INFO - ['val_loss: 0.143050']
Epoch 3/5: 100%|██████████| 500/500 [00:37<00:00, 13.24it/s, loss=0.145688]
2025-10-11 13:06:19,516 - flow-matching - INFO - ['val_loss: 0.140311']
Epoch 4/5: 100%|██████████| 500/500 [00:37<00:00, 13.26it/s, loss=0.140565]
2025-10-11 13:06:58,155 - flow-matching - INFO - ['val_loss: 0.141859']


In [5]:
torch.save(backbone.state_dict(), "./models/backbone_flow.pt")